# Fine-Tune the LingBot-VLA-v2 Action Expert: Four-GPU RoboTwin Workshop

Start from the official base policy and fine-tune its action expert on 50 clean `turn_switch` demonstrations. Compare the base policy, the trained action expert, and the **official RoboTwin-trained checkpoint** on the same clean and randomized seeds. The final section records and displays one successful rollout from the official RoboTwin checkpoint.

Run the cells in order in the prepared RoboTwin workshop environment. Training and evaluation use four GPUs.


## What this workshop uses

| Component | Role in this workshop |
|---|---|
| Policy | LingBot-VLA-v2-6B takes camera images, robot state, and a task instruction and predicts an action sequence. |
| Simulator | RoboTwin uses SAPIEN scenes, MPLib planning, ROCm model computation, and RADV/Vulkan rendering. |
| Task | Learn `turn_switch` from 50 `demo_clean` demonstrations; evaluate 10 fixed seeds in both `demo_clean` and `demo_randomized`. |
| Hardware | Four idle AMD ROCm GPUs. FSDP2 shards training; each evaluation worker runs a model service and simulator on one GPU. |
| Software | RoboTwin, LingBot-VLA-v2, PyTorch ROCm, SAPIEN, MPLib, and LeRobot. The next section prints their actual paths and versions. |

Each evaluation seed identifies one closed-loop episode. The policy repeatedly observes, predicts, and acts until the task succeeds or reaches its step limit. Clean and randomized runs share the seed list but use different scene settings and instruction categories. Ten episodes per scene are a workshop check; use more valid rollouts for a formal success rate.


## Watch the task first

In `turn_switch`, the robot approaches the switch with the appropriate arm and moves its lever to the target end position. The task succeeds when the switch reaches that position. The video shows one expert demonstration of the task.


In [1]:
from IPython.display import HTML, display

_TASK_VIDEO_B64 = (
    '...'
    'NTguNzYuMTAw'
)
display(HTML(
    '<video controls preload="metadata" width="480">'
    '<source src="data:video/mp4;base64,' + _TASK_VIDEO_B64 + '" type="video/mp4">'
    'Your browser does not support embedded video.'
    '</video>'
))


turn_switch expert demonstration video

## Background: the model and the weights we update

LingBot-VLA-v2 connects a vision-language backbone to an action expert to predict robot actions from images, state, and instructions. This workshop initializes from the official base checkpoint. It sets `use_lora=false`, `train_expert_only=true`, and `freeze_vision_encoder=true`. In the pinned LingBot implementation, expert-only mode freezes the Qwen vision-language backbone; the action expert and related action-side layers remain trainable. This is supervised fine-tuning, **not full-parameter SFT**. FSDP2 distributes the workload across four GPUs. The short run uses neither depth/video teachers nor LoRA adapters.

The official RoboTwin-trained checkpoint used a different recipe. A later cell reads its `lingbotvla_cli.yaml` and compares steps, batch size, optimizer, and teacher settings. **This 100-step, single-task run is for learning and workflow validation; it is not expected to match the official checkpoint's success rate.**


## Approximate time budget

Workflow: **task video → environment check → recipe comparison → base evaluation → action-expert SFT → loss curve → checkpoint conversion → trained-policy evaluation → official checkpoint evaluation → official success video**.

Plan for **about 40–45 minutes** for the complete workshop. The schedule includes task and environment explanations, hands-on checks, model runs, result discussion, and the final video. Explain the model and evaluation concepts while the longer GPU cells run.

| Part | Planned time | What to do |
|---|---:|---|
| Task video, policy, and simulator concepts | 3 min | Explain the goal and the closed-loop setup. |
| Environment, GPU, data, and recipe preflight | 4 min | Walk through the checks and key settings. |
| Base policy evaluation | 6–7 min | Run evaluation and explain the live metrics. |
| Action-expert training and loss curve | 10 min | Run training and discuss parameters and loss. |
| Checkpoint conversion | 1 min | Prepare inference weights. |
| Trained-policy evaluation | 6–7 min | Compare outcomes with the base policy. |
| Official-checkpoint evaluation | 6–7 min | Add the reference model. |
| Result comparison and official success video | 5 min | Discuss the table and play the video. |
| **Total** | **about 40–45 min** | |

The live panels show completed seeds, current success rate, and estimated remaining time during the training and evaluation blocks. At least 100 GiB of free output space is required.


## 1. Check software locations and versions

The workshop environment is preconfigured. This cell prints the actual Python, ROCm, simulator, data, and model locations and versions.


In [ ]:
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import os, shutil, subprocess, sys

if Path(sys.prefix) != Path('/opt/robotwin-env'):
    raise RuntimeError(f'The prepared RoboTwin Python environment is unavailable: {sys.executable}')

paths = {
    'Python': sys.executable,
    'RoboTwin': '/RoboTwin',
    'LingBot source': '/RoboTwin/experiments/lingbot_vla_v2_6b_robotwin/source/lingbot-vla-v2',
    'Dataset': '/RoboTwin/data/lerobot/turn_switch_joint_v30',
    'Base checkpoint': '/RoboTwin/experiments/lingbot_vla_v2_6b_robotwin/models/robbyant_lingbot-vla-v2-6b',
    'Official RoboTwin checkpoint': '/RoboTwin/experiments/lingbot_vla_v2_6b_robotwin/models/robbyant_lingbot-vla-v2-6b-robotwin',
    'Runtime outputs': '/workspace/runtime/outputs',
}
for label, location in paths.items():
    print(f'{label:28} {location}')
for package in ('torch', 'triton', 'sapien', 'mplib', 'lerobot', 'jupyterlab'):
    try:
        value = version(package)
    except PackageNotFoundError:
        value = 'missing'
    print(f'{package:28} {value}')
print(f'{"rocm-smi":28} {shutil.which("rocm-smi")}')
print(f'{"ffmpeg":28} {shutil.which("ffmpeg")}')
subprocess.run(['rocm-smi', '--showproductname'], check=True)


## 2. Select GPUs and define this run

Select four GPUs with free VRAM. Evaluation uses 10 fixed seeds and four concurrent workers: two for clean and two for randomized, with five seeds per worker. All four GPUs contribute to each training global batch.


In [ ]:
from datetime import datetime, timezone
from IPython.display import Markdown, Video, clear_output, display
import hashlib, json, re, signal, time

PROJECT_ROOT = Path('/RoboTwin')
EXP = PROJECT_ROOT / 'experiments/lingbot_vla_v2_6b_robotwin'
SCRIPT_ROOT = EXP / 'scripts'
SOURCE = EXP / 'source/lingbot-vla-v2'
BASE = EXP / 'models/robbyant_lingbot-vla-v2-6b'
OFFICIAL_ROOT = EXP / 'models/robbyant_lingbot-vla-v2-6b-robotwin'
OFFICIAL_MODEL = OFFICIAL_ROOT / 'checkpoints/global_step_50000/hf_ckpt'
OFFICIAL_CONFIG = OFFICIAL_ROOT / 'lingbotvla_cli.yaml'
YAML = EXP / 'training/lingbotvla_cli.yaml'
EVALUATOR = SCRIPT_ROOT / 'eval/paired_seed_evaluator.py'
sys.path.insert(0, str(SCRIPT_ROOT / 'eval'))

selection = json.loads(subprocess.check_output(
    [sys.executable, str(SCRIPT_ROOT / 'eval/select_idle_gpus.py'), '--count', '4'],
    text=True))
GPU_IDS = selection['gpu_ids']
GPU_CARDS = selection['gpu_cards']
TASK_NAME = 'turn_switch'
SEEDS = [100000, 100001, 100002, 100005, 100007,
         100009, 100010, 100011, 100012, 100013]
PORT = 13691
STEPS = 100
MICRO_BATCH = 8
GRAD_ACCUM = 1
GLOBAL_BATCH = len(GPU_IDS) * MICRO_BATCH * GRAD_ACCUM
LEARNING_RATE = 5e-5
CAMERA_NEAR = 0.01
os.environ['ROBOTWIN_CAMERA_NEAR'] = str(CAMERA_NEAR)
stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
RUN_NAME = f'guided_action_expert_{TASK_NAME}_gpus{"-".join(map(str, GPU_IDS))}_{stamp}'
RUN_DIR = Path('/workspace/runtime/outputs/robotwin-action-expert-workshop-runs') / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
DATA = PROJECT_ROOT / f'data/lerobot/{TASK_NAME}_joint_v30'
CHECKPOINT = RUN_DIR / f'checkpoints/global_step_{STEPS}'
MERGED = RUN_DIR / f'merged_checkpoint/global_step_{STEPS}/hf_ckpt'
TIMING_PATH = RUN_DIR / 'stage_times.json'
stage_times = json.loads(TIMING_PATH.read_text()) if TIMING_PATH.exists() else {}

def save_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n')

def record_time(stage, started):
    stage_times[stage] = {'seconds': round(time.monotonic() - started, 1)}
    save_json(TIMING_PATH, stage_times)

def gib(value):
    return f'{value / 2**30:,.1f} GiB'

print('run:', RUN_DIR)
print('task:', TASK_NAME, '| steps:', STEPS, '| seeds per scene:', len(SEEDS))
print('global batch:', GLOBAL_BATCH, '=', len(GPU_IDS), 'GPUs ×', MICRO_BATCH,
      'micro batch ×', GRAD_ACCUM, 'accumulation')
for gpu in GPU_IDS:
    card = GPU_CARDS[str(gpu)]
    print(f'GPU {gpu}: {card["model"]} ({card["arch"]}), '
          f'used {card["vram_used_gib"]:.2f} GiB, {card["dri_prime"]}')


## 3. Check models, data, configuration, and space

Check that the base checkpoint, 50 clean task demonstrations, training configuration, evaluation scripts, four GPUs, and output space are ready before running the workshop.


In [ ]:
started = time.monotonic()
required = (BASE / 'model.safetensors.index.json', DATA / 'meta/info.json', YAML,
            SOURCE / 'deploy/lingbot_vla_v2_policy.py', EVALUATOR,
            SCRIPT_ROOT / 'training/train_full_sft.sh',
            SCRIPT_ROOT / 'checkpoints/convert_full_sft_dcp.py',
            SCRIPT_ROOT / 'eval/run_parallel_paired_eval.py',
            OFFICIAL_MODEL / 'model.safetensors.index.json', OFFICIAL_CONFIG)
for path in required:
    if not path.exists():
        raise FileNotFoundError(path)
launcher_source = (SCRIPT_ROOT / 'training/train_full_sft.sh').read_text()
if 'TRAIN_EXPERT_ONLY' in launcher_source and 'FREEZE_VISION_ENCODER' in launcher_source:
    TRAIN_LAUNCHER = SCRIPT_ROOT / 'training/train_full_sft.sh'
else:
    # Keep the image source untouched; enable expert-only mode in this run's copy.
    lines = launcher_source.splitlines()
    matches = [index for index, line in enumerate(lines)
               if '--train.train_expert_only false' in line]
    if len(matches) != 1:
        raise RuntimeError('Cannot locate the expert-only flag in the training launcher')
    index = matches[0]
    lines[index] = lines[index].replace('false', 'true', 1)
    lines.insert(index + 1, '  --train.freeze_vision_encoder true ' + chr(92))
    TRAIN_LAUNCHER = RUN_DIR / 'train_action_expert.sh'
    TRAIN_LAUNCHER.write_text('\n'.join(lines) + '\n')
    TRAIN_LAUNCHER.chmod(0o755)
parallel_source = (SCRIPT_ROOT / 'eval/run_parallel_paired_eval.py').read_text()
if 'choices=("baseline", "sft", "robotwin")' in parallel_source:
    PARALLEL_EVAL = SCRIPT_ROOT / 'eval/run_parallel_paired_eval.py'
else:
    # Extend the committed parallel helper in this run's directory only.
    needle = 'choices=("baseline", "sft")'
    if parallel_source.count(needle) != 1:
        raise RuntimeError('Cannot enable the official checkpoint in the parallel evaluator')
    parallel_source = parallel_source.replace(
        needle, 'choices=("baseline", "sft", "robotwin")', 1)
    order = '("baseline_clean", "baseline_random", "sft_clean", "sft_random")'
    parallel_source = parallel_source.replace(
        order,
        '("baseline_clean", "baseline_random", "sft_clean", "sft_random", '
        '"robotwin_clean", "robotwin_random")', 1)
    helpers = RUN_DIR / 'eval_helpers'
    helpers.mkdir(exist_ok=True)
    PARALLEL_EVAL = helpers / 'run_parallel_paired_eval.py'
    PARALLEL_EVAL.write_text(parallel_source)
    for name in ('run_paired_eval.py', 'video_files.py', 'resolve_dri_prime.py'):
        link = helpers / name
        if not link.exists():
            link.symlink_to(SCRIPT_ROOT / 'eval' / name)
base_index_sha = hashlib.sha256((BASE / 'model.safetensors.index.json').read_bytes()).hexdigest()
yaml_sha = hashlib.sha256(YAML.read_bytes()).hexdigest()
if base_index_sha != '5a753ec331c51925d064e1e76e921a7eb3dca9770d2a6a91dac5e0e4162d676a':
    raise RuntimeError('Unexpected official base checkpoint index')
if yaml_sha != '808a2718c81595e0303fa8b5d4b563d004014395711ae9cf371acd1a0ab635df':
    raise RuntimeError('Unexpected training YAML')
info = json.loads((DATA / 'meta/info.json').read_text())
if info['total_episodes'] != 50:
    raise RuntimeError(f'Expected 50 clean {TASK_NAME} demonstrations')
if len(SEEDS) != 10 or len(SEEDS) != len(set(SEEDS)) or len(GPU_IDS) != 4:
    raise RuntimeError('Need 10 unique paired seeds and four GPUs')
free_disk = shutil.disk_usage(RUN_DIR).free
if free_disk < 100 * 2**30:
    raise RuntimeError(f'This checkpoint workflow requires 100 GiB free; found {gib(free_disk)}')
import torch, triton, sapien
manifest = {
    'mode': 'guided_four_gpu_action_expert_sft', 'run_name': RUN_NAME,
    'task': TASK_NAME, 'gpu_ids': GPU_IDS, 'gpu_cards': GPU_CARDS,
    'dataset_episodes': info['total_episodes'], 'dataset_frames': info['total_frames'],
    'base_index_sha256': base_index_sha, 'training_yaml_sha256': yaml_sha,
    'seeds': SEEDS, 'steps': STEPS, 'micro_batch': MICRO_BATCH,
    'gradient_accumulation': GRAD_ACCUM, 'global_batch': GLOBAL_BATCH,
    'optimizer': 'adamw', 'learning_rate': LEARNING_RATE,
    'teacher_mode': 'none', 'fsdp_cpu_offload': False,
    'train_expert_only': True, 'freeze_vision_encoder': True,
    'torch': torch.__version__, 'hip': torch.version.hip,
    'triton': triton.__version__, 'sapien': sapien.__version__,
}
save_json(RUN_DIR / 'manifest.json', manifest)
save_json(RUN_DIR / 'paired_eval_seeds.json', SEEDS)
(RUN_DIR / 'task_only.txt').write_text(f'robotwin {DATA}\n')
record_time('preflight', started)
print(f'Data: {info["total_episodes"]} demonstrations, {info["total_frames"]} frames')
print('Free output:', gib(free_disk), f'({free_disk / 2**40:.2f} TiB)')
print('Official RoboTwin checkpoint:', OFFICIAL_MODEL)


## 4. Compare this action-expert run with the official RoboTwin recipe

The right column comes from the YAML shipped with the official checkpoint. This workshop updates the action expert using 50 clean demonstrations from one task, 100 AdamW steps, and ordinary supervised action loss. The official configuration records broader training, more steps, and depth/video alignment. Their results provide a reference, not a controlled ablation.


In [ ]:
import yaml
official = yaml.safe_load(OFFICIAL_CONFIG.read_text())
official_train = official['train']
official_data = official['data']
official_model = official['model']
def on_off(value):
    if value is None:
        return 'not recorded'
    return 'on' if value else 'off'

official_shards = int(official_train['data_parallel_shard_size'])
official_replicas = int(official_train.get('data_parallel_replicate_size', 1))
official_gpu_count = official_shards * official_replicas
comparisons = [
    ('Training data', f'{TASK_NAME}: 50 clean demonstrations', str(official_data.get('train_path'))),
    ('Training GPUs / ranks', len(GPU_IDS), f'{official_gpu_count} (implied by YAML)'),
    ('Data-parallel replica groups', 1, official_replicas),
    ('Data-parallel shard size', len(GPU_IDS), official_shards),
    ('Optimizer steps', STEPS, official_train.get('max_steps')),
    ('Optimizer', 'AdamW', official_train.get('optimizer')),
    ('Learning rate', LEARNING_RATE, official_train.get('lr')),
    ('Global batch', GLOBAL_BATCH, official_train.get('global_batch_size')),
    ('Micro batch per GPU', MICRO_BATCH, official_train.get('micro_batch_size')),
    ('Depth/video teacher', 'off', on_off(bool(official_train.get('align_params')))),
    ('Post-training alignment', 'off', on_off(official_model.get('post_training'))),
    ('Future image', 'off', on_off(official_data.get('use_future_image'))),
    ('FSDP full shard', 'on', on_off(official_train.get('enable_full_shard'))),
    ('CPU offload', 'off', on_off(official_train.get('enable_fsdp_offload'))),
    ('LoRA', 'off', on_off(official_train['use_lora']) if 'use_lora' in official_train
     else 'not listed (default off)'),
    ('Expert-only training', 'on', on_off(official_train.get('train_expert_only'))),
    ('Freeze vision encoder', 'on', on_off(official_train.get('freeze_vision_encoder'))),
]
lines = ['| Parameter | This workshop | Official checkpoint YAML |', '|---|---|---|']
lines += [f'| {name} | {workshop} | {upstream} |'
          for name, workshop, upstream in comparisons]
display(Markdown('\n'.join(lines)))


## 5. Enable live progress panels

Live panels show training steps, completed evaluation episodes, current success rate, and estimated time remaining.


In [ ]:
def tail_text(path, limit=131072):
    if not path.is_file():
        return ''
    with path.open('rb') as stream:
        stream.seek(max(0, path.stat().st_size - limit))
        return stream.read().decode(errors='replace')

def read_rows(path):
    rows = []
    if path.is_file():
        for line in path.read_text(errors='replace').splitlines():
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass  # A worker may still be appending the last line.
    return rows

def duration(seconds):
    if seconds is None:
        return 'waiting for first sample'
    seconds = max(0, int(seconds))
    return f'{seconds // 3600}h {seconds % 3600 // 60}m' if seconds >= 3600 else f'{seconds // 60}m {seconds % 60}s'

def run_monitored(command, log_path, title, snapshot, *, env=None):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    with log_path.open('w') as output:
        process = subprocess.Popen(command, cwd=SCRIPT_ROOT, env=env,
                                   stdout=output, stderr=subprocess.STDOUT,
                                   start_new_session=True)
        try:
            while process.poll() is None:
                clear_output(wait=True)
                display(Markdown(f'### {title}\n\n' + snapshot(time.monotonic() - started)))
                time.sleep(10)
            code = process.wait()
        except KeyboardInterrupt:
            os.killpg(process.pid, signal.SIGINT)
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
            raise
    clear_output(wait=True)
    display(Markdown(f'### {title}\n\n' + snapshot(time.monotonic() - started)))
    if code:
        print('\n'.join(tail_text(log_path, 12000).splitlines()[-25:]))
        raise RuntimeError(f'{title} failed (rc={code}); see {log_path}')
    print(f'{title} complete in {duration(time.monotonic() - started)}; log: {log_path}')

def evaluation_snapshot(method, elapsed):
    root = RUN_DIR / 'evaluations/no_video'
    table = ['| Scene / GPU | Complete | Current seed | Success so far | ETA |',
             '|---|---:|---|---:|---:|']
    totals = {'clean': [0, 0], 'random': [0, 0]}
    worker_etas = []
    for scene, gpu, assigned in (
        ('clean', GPU_IDS[0], SEEDS[::2]), ('clean', GPU_IDS[1], SEEDS[1::2]),
        ('random', GPU_IDS[2], SEEDS[::2]), ('random', GPU_IDS[3], SEEDS[1::2])):
        label = f'{method}_{scene}'
        shard = root / 'shards' / f'{label}_gpu{gpu}'
        rows = read_rows(shard / f'{label}_episodes.jsonl')
        done = min(len(rows), len(assigned))
        successes = sum(bool(row.get('success')) for row in rows[:done])
        totals[scene][0] += done
        totals[scene][1] += successes
        if done == len(assigned):
            current = 'done'
            eta = 0
        else:
            serving = done > 0 or 'server listening' in tail_text(shard / f'{label}_server.log', 50000)
            current = (f'{done + 1}/{len(assigned)}: seed {assigned[done]}' if serving
                       else f'loading model → seed {assigned[done]}')
            samples = [float(row['total_s']) for row in rows if row.get('total_s')]
            eta = ((len(assigned) - done) * sum(samples) / len(samples)) if samples else None
        if eta is not None:
            worker_etas.append(eta)
        rate = f'{successes}/{done} ({successes / done:.0%})' if done else '—'
        table.append(f'| {scene} / GPU {gpu} | {done}/{len(assigned)} | {current} | {rate} | {duration(eta)} |')
    line = []
    for scene in ('clean', 'random'):
        done, successes = totals[scene]
        line.append(f'{scene}: {done}/{len(SEEDS)} complete, current success rate '
                    f'{successes / done:.0%}' if done else f'{scene}: 0/{len(SEEDS)} complete')
    overall_eta = max(worker_etas) if len(worker_etas) == 4 else None
    return (f'Elapsed {duration(elapsed)}; estimated remaining {duration(overall_eta)} '
            '(roughly based on completed episodes).\n\n' + '; '.join(line) + '\n\n' + '\n'.join(table))

def evaluate(method):
    started = time.monotonic()
    command = [sys.executable, str(PARALLEL_EVAL),
               '--run-dir', str(RUN_DIR), '--result-tag', 'no_video',
               '--gpu-ids', ','.join(map(str, GPU_IDS)), '--port-base', str(PORT),
               '--count', str(len(SEEDS)), '--steps', str(STEPS),
               '--task-name', TASK_NAME, '--method', method]
    label = 'action-expert SFT' if method == 'sft' else method
    run_monitored(command, RUN_DIR / f'evaluate_{method}_no_video.log',
                  f'{label}: clean + randomized evaluation',
                  lambda elapsed: evaluation_snapshot(method, elapsed))
    record_time(f'evaluate_{method}_no_video', started)
    return json.loads((RUN_DIR / 'evaluations/no_video/summary.json').read_text())

def training_snapshot(elapsed):
    config = (f'Configuration: {len(GPU_IDS)} GPUs × micro {MICRO_BATCH} × accumulation {GRAD_ACCUM} '
              f'= global {GLOBAL_BATCH}; AdamW LR {LEARNING_RATE:g}; '
              'action expert only, FSDP2 full shard, mixed precision, '
              'no CPU offload, no depth/video teacher.\n\n')
    text = tail_text(RUN_DIR / 'train.log')
    matches = list(re.finditer(
        r'Step\s+(\d+)/\d+,\s+Epoch\s+\d+,\s+Loss\s+([\d.]+).*?'
        r'LR\s+([\deE+.-]+).*?StepTime\s+([\d.]+)s', text))
    if not matches:
        return config + f'Elapsed {duration(elapsed)}; initializing the model, data, or first step. Training log: `{RUN_DIR / "train.log"}`'
    last = matches[-1]
    step = min(int(last.group(1)), STEPS)
    times = [float(match.group(4)) for match in matches[-5:]]
    eta = (STEPS - step) * sum(times) / len(times)
    phase = 'saving the DCP checkpoint' if step == STEPS else 'optimizer training'
    return (config + f'**{phase}**; step **{step}/{STEPS}**; loss **{last.group(2)}**; '
            f'LR **{last.group(3)}**; latest step **{times[-1]:.1f}s**.\n\n'
            f'Elapsed {duration(elapsed)}; roughly {duration(eta)} remaining plus checkpoint save time.')

def conversion_snapshot(elapsed):
    files = list(MERGED.glob('*.safetensors')) if MERGED.exists() else []
    size = sum(path.stat().st_size for path in files)
    index = (MERGED / 'model.safetensors.index.json').is_file()
    phase = ('index written; checking output' if index else
             'writing Hugging Face weights' if MERGED.exists() else
             'reading and merging DCP shards')
    return (f'**{phase}**; elapsed {duration(elapsed)}; '
            f'{len(files)} safetensors files written, totaling {gib(size)}.\n\n'
            'Conversion has no per-layer counter; completion time depends on disk speed. See `convert.log` for details.')

policy_file = SOURCE / 'deploy/lingbot_vla_v2_policy.py'
if 'ROBOTWIN_WORKSHOP_DISABLE_FUSED_MOE' not in policy_file.read_text():
    raise RuntimeError('Missing workshop MoE compatibility patch')
if '"seed": int(now_seed)' not in EVALUATOR.read_text():
    raise RuntimeError('Missing per-episode policy seed support')
print('Progress dashboard ready. Evaluations will reuse the same policy and evaluator.')


## 6. Evaluate the base policy (~7 min)

Run the same 10 fixed seeds in clean and randomized scenes on four GPUs, without video. The panel shows each worker's current seed and success rate among completed episodes.


In [ ]:
baseline_results = evaluate('baseline')
print('baseline:', baseline_results)


## 7. Fine-tune the action expert on four GPUs (~10 min)

Review the actual overrides before launching. The launcher sets `TRAIN_EXPERT_ONLY=true` and `FREEZE_VISION_ENCODER=true`, freezing the Qwen vision-language backbone. It uses FSDP2 full sharding, mixed precision, and gradient checkpointing, with **CPU offload disabled**. `TEACHER_MODE=none` skips depth/video teachers. After the first optimizer step, the panel shows step, loss, learning rate, step time, and ETA.


In [ ]:
train_params = {
    'task': TASK_NAME, 'demonstrations': info['total_episodes'],
    'GPU IDs': GPU_IDS, 'FSDP shard size': len(GPU_IDS),
    'optimizer': 'AdamW', 'learning rate': LEARNING_RATE,
    'steps': STEPS, 'micro batch per GPU': MICRO_BATCH,
    'gradient accumulation': GRAD_ACCUM, 'global batch': GLOBAL_BATCH,
    'LoRA': False, 'expert only': True, 'freeze vision encoder': True,
    'teacher mode': 'none',
    'CPU offload': False, 'checkpoint': str(CHECKPOINT),
}
display(Markdown('### Training parameters\n\n' + '\n'.join(
    ['| Parameter | Value |', '|---|---|'] +
    [f'| {name} | {value} |' for name, value in train_params.items()])))
log = RUN_DIR / 'train.log'
complete = ((CHECKPOINT / 'model/.metadata').is_file() and
            (CHECKPOINT / 'optimizer/.metadata').is_file() and log.is_file() and
            f'Reached max_steps={STEPS}' in log.read_text(errors='replace'))
if complete:
    print('Using completed action-expert checkpoint:', CHECKPOINT)
else:
    if CHECKPOINT.exists():
        raise RuntimeError(f'Incomplete SFT checkpoint: {CHECKPOINT}')
    env = os.environ.copy()
    for key in ('ROCR_VISIBLE_DEVICES', 'CUDA_VISIBLE_DEVICES', 'GPU_DEVICE_ORDINAL'):
        env.pop(key, None)
    env.update({
        'GPU_COUNT': str(len(GPU_IDS)), 'GPU_IDS': ','.join(map(str, GPU_IDS)),
        'MICRO_BATCH_SIZE': str(MICRO_BATCH), 'GLOBAL_BATCH_SIZE': str(GLOBAL_BATCH),
        'GRADIENT_ACCUMULATION_STEPS': str(GRAD_ACCUM),
        'MAX_STEPS': str(STEPS), 'SAVE_STEPS': str(STEPS),
        'OUTPUT_DIR': str(RUN_DIR), 'LOG_FILE': str(log),
        'SFT_LR': str(LEARNING_RATE), 'DATA_LIST': str(RUN_DIR / 'task_only.txt'),
        'TEACHER_MODE': 'none', 'OPTIMIZER': 'adamw',
        'TRAIN_EXPERT_ONLY': 'true', 'FREEZE_VISION_ENCODER': 'true',
        'NUM_WORKERS': '0', 'PIN_MEMORY': 'false', 'IMAGE_AUGMENT': 'false',
        'ENABLE_FSDP_OFFLOAD': 'false', 'ENABLE_ACTIVATION_OFFLOAD': 'false',
        'ENABLE_FORWARD_PREFETCH': 'false', 'EMPTY_CACHE_STEPS': '1',
        'ENABLE_FULL_SHARD': 'true', 'DATA_PARALLEL_MODE': 'fsdp2',
        'AITER_USE_SYSTEM_TRITON': '1', 'AITER_TRITON_ONLY': '1',
        'FLASH_ATTENTION_TRITON_AMD_ENABLE': 'TRUE',
        'SETUPTOOLS_SCM_PRETEND_VERSION': '0.0.0',
        'PYTORCH_ALLOC_CONF': 'expandable_segments:True',
    })
    started = time.monotonic()
    run_monitored(['bash', str(TRAIN_LAUNCHER)],
                  RUN_DIR / 'train_launcher.log', 'Four-GPU action-expert SFT training', training_snapshot, env=env)
    if not ((CHECKPOINT / 'model/.metadata').is_file() and
            (CHECKPOINT / 'optimizer/.metadata').is_file()):
        raise RuntimeError(f'SFT checkpoint is incomplete: {CHECKPOINT}')
    if f'Reached max_steps={STEPS}' not in log.read_text(errors='replace'):
        raise RuntimeError(f'SFT did not reach step {STEPS}; see {log}')
    record_time('train', started)
print('checkpoint:', CHECKPOINT)


### Inspect the training loss curve

The trainer logs total loss and VLA action loss at each optimizer step. The next cell plots both training loss curves.


In [ ]:
import matplotlib.pyplot as plt

loss_pattern = re.compile(
    r'Step\s+(\d+)/\d+,\s+Epoch\s+\d+,\s+Loss\s+([\deE+.-]+),\s+'
    r'VLA_Loss\s+([\deE+.-]+)')
loss_by_step = {int(match.group(1)): (float(match.group(2)), float(match.group(3)))
                for match in loss_pattern.finditer(log.read_text(errors='replace'))}
if len(loss_by_step) < STEPS:
    raise RuntimeError(f'Expected {STEPS} logged training steps; found {len(loss_by_step)} in {log}')
steps = sorted(loss_by_step)
total_loss = [loss_by_step[step][0] for step in steps]
action_loss = [loss_by_step[step][1] for step in steps]
fig, (total_ax, action_ax) = plt.subplots(2, 1, figsize=(9, 6), sharex=True, sharey=True)
total_ax.plot(steps, total_loss, color='#1f77b4', linewidth=1.5)
total_ax.set(ylabel='Total loss', title='Action-expert SFT: raw training losses')
action_ax.plot(steps, action_loss, color='#ff7f0e', linewidth=1.5)
action_ax.set(xlabel='Optimizer step', ylabel='VLA action loss')
for ax in (total_ax, action_ax):
    ax.grid(alpha=0.25)
fig.tight_layout()
loss_plot = RUN_DIR / 'training_loss.png'
fig.savefig(loss_plot, dpi=150)
plt.show()
print('saved loss curve:', loss_plot)
print('mean Total − VLA loss:', sum(a - b for a, b in zip(total_loss, action_loss)) / len(steps))


## 8. Convert the distributed checkpoint for inference

The training DCP contains model and optimizer shards. The model service needs Hugging Face-format weights. The conversion panel shows the current phase and the number and size of written weight files. The cell checks the index file at the end.


In [ ]:
if (MERGED / 'model.safetensors.index.json').is_file():
    print('Using completed inference checkpoint:', MERGED)
else:
    if MERGED.exists():
        raise RuntimeError(f'Incomplete converted model: {MERGED}')
    env = os.environ.copy()
    for key in ('ROCR_VISIBLE_DEVICES', 'CUDA_VISIBLE_DEVICES', 'GPU_DEVICE_ORDINAL'):
        env.pop(key, None)
    env['HIP_VISIBLE_DEVICES'] = str(GPU_IDS[0])
    started = time.monotonic()
    command = [sys.executable, str(SCRIPT_ROOT / 'checkpoints/convert_full_sft_dcp.py'),
               '--checkpoint', str(CHECKPOINT), '--training-output', str(RUN_DIR),
               '--output', str(MERGED)]
    run_monitored(command, RUN_DIR / 'convert.log', 'DCP → HF checkpoint conversion',
                  conversion_snapshot, env=env)
    if not (MERGED / 'model.safetensors.index.json').is_file():
        raise RuntimeError(f'Conversion did not create {MERGED}')
    record_time('convert', started)
print('inference model:', MERGED)


## 9. Evaluate the trained action expert (~7 min)

Use the same 10 clean and 10 randomized seeds as the base policy. The current success rate updates as episodes finish. All per-episode records remain in the run directory.


In [ ]:
sft_results = evaluate('sft')
print('action-expert SFT:', sft_results)


## 10. Evaluate the official RoboTwin checkpoint (~7 min)

Load the prepared official `global_step_50000/hf_ckpt` under `/models` with its accompanying training YAML. Evaluate the same 10 clean and 10 randomized seeds on four GPUs, without video. The official checkpoint used different data and a different recipe, so this is a reference comparison rather than a single-variable experiment.


In [ ]:
official_results = evaluate('robotwin')
print('official RoboTwin:', official_results)


## 11. Compare results and elapsed time

Compare success counts for all three policies on the same ten seeds. This is a workshop comparison; use more independently chosen seeds for a formal evaluation. The next cell preserves per-seed outcomes and stage durations.


In [ ]:
summary = json.loads((RUN_DIR / 'evaluations/no_video/summary.json').read_text())
lines = ['| Scene | Base policy | 100-step action-expert SFT | Official RoboTwin checkpoint |',
         '|---|---:|---:|---:|']
for scene in ('clean', 'random'):
    lines.append(f'| {scene} | {summary[f"baseline_{scene}"]}/{len(SEEDS)} | '
                 f'{summary[f"sft_{scene}"]}/{len(SEEDS)} | '
                 f'{summary[f"robotwin_{scene}"]}/{len(SEEDS)} |')
display(Markdown('\n'.join(lines)))
for scene in ('clean', 'random'):
    all_rows = {}
    for method in ('baseline', 'sft', 'robotwin'):
        path = RUN_DIR / 'evaluations/no_video' / f'{method}_{scene}_episodes.jsonl'
        rows = read_rows(path)
        if len(rows) != len(SEEDS) or [int(row['seed']) for row in rows] != SEEDS:
            raise RuntimeError(f'Incomplete {method} {scene} result: {path}')
        all_rows[method] = rows
    detail = [f'### {scene}: per-seed outcomes', '',
              '| Seed | Base | SFT | Official checkpoint |', '|---:|:---:|:---:|:---:|']
    for index, seed in enumerate(SEEDS):
        outcomes = [('✓' if all_rows[method][index]['success'] else '×')
                    for method in ('baseline', 'sft', 'robotwin')]
        detail.append(f'| {seed} | ' + ' | '.join(outcomes) + ' |')
    display(Markdown('\n'.join(detail)))
timing_lines = ['| Stage | Minutes |', '|---|---:|']
for stage in ('preflight', 'evaluate_baseline_no_video', 'train', 'convert',
              'evaluate_sft_no_video', 'evaluate_robotwin_no_video'):
    if stage in stage_times:
        timing_lines.append(f'| {stage} | {stage_times[stage]["seconds"] / 60:.1f} |')
display(Markdown('\n'.join(timing_lines)))
print('run directory:', RUN_DIR)


## 12. Record one successful official RoboTwin video (~3 min)

Display only a successful recorded rerun. This cell reloads the model before recording.

RoboTwin's default video misses much of the motion between actions, so the next two cells enable continuous recording when needed.


In [ ]:
ROBOTWIN_VIDEO_PATCH = r'''--- a/envs/_base_task.py
+++ b/envs/_base_task.py
@@ -585,5 +585,20 @@
     def _set_eval_video_ffmpeg(self, ffmpeg):
         self.eval_video_ffmpeg = ffmpeg
-
+        self._eval_video_sim_steps = 0
+
+    def _write_eval_video_frame(self):
+        if self.eval_video_path is None or self.eval_video_ffmpeg is None:
+            return
+        frame = self.now_obs["observation"]["head_camera"]["rgb"]
+        self.eval_video_ffmpeg.stdin.write(frame.tobytes())
+
+    def _maybe_write_eval_video_frame(self):
+        if self.eval_video_path is None or self.eval_video_ffmpeg is None:
+            return
+        self._eval_video_sim_steps += 1
+        if self._eval_video_sim_steps % 10 == 0:
+            self.get_obs()
+            self._write_eval_video_frame()
+
     def close_env(self, clear_cache=False):
         if clear_cache:
@@ -1493,6 +1508,2 @@
-        eval_video_freq = 1  # fixed
-        if (self.eval_video_path is not None and self.take_action_cnt % eval_video_freq == 0):
-            self.eval_video_ffmpeg.stdin.write(self.now_obs["observation"]["head_camera"]["rgb"].tobytes())
-
         self.take_action_cnt += 1
         print(f"step: \033[92m{self.take_action_cnt} / {self.step_lim}\033[0m", end="\r")
@@ -1667,6 +1678,6 @@
+            self._maybe_write_eval_video_frame()
             if self.check_success():
                 self.eval_success = True
-                self.get_obs() # update obs
-                if (self.eval_video_path is not None):
-                    self.eval_video_ffmpeg.stdin.write(self.now_obs["observation"]["head_camera"]["rgb"].tobytes())
+                self.get_obs()  # final success frame
+                self._write_eval_video_frame()
                 return
@@ -1674,3 +1685,6 @@
         self._update_render()
+        if self.eval_video_path is not None:
+            self.get_obs()  # final frame after this action
+            self._write_eval_video_frame()
         if self.render_freq:  # UI
             self.viewer.render()
'''

EVALUATOR_VIDEO_PATCH = r'''--- a/scripts/eval/paired_seed_evaluator.py
+++ b/scripts/eval/paired_seed_evaluator.py
@@ -687,7 +687,7 @@
                 "-video_size",
                 video_size,
                 "-framerate",
-                "10",
+                "25",
                 "-i",
                 "-",
                 "-pix_fmt",
@@ -701,6 +701,8 @@
             stdin=subprocess.PIPE,
         )
         task_env._set_eval_video_ffmpeg(ffmpeg)
+        task_env.get_obs()
+        task_env._write_eval_video_frame()

     succ = False
     inference_latencies = []
@@ -949,7 +951,7 @@
                     "-video_size",
                     video_size,
                     "-framerate",
-                    "10",
+                    "25",
                     "-i",
                     "-",
                     "-pix_fmt",
@@ -963,6 +965,8 @@
                 stdin=subprocess.PIPE,
             )
             task_env._set_eval_video_ffmpeg(ffmpeg)
+            task_env.get_obs()
+            task_env._write_eval_video_frame()

         succ = False
         inference_latencies = []
'''


In [ ]:
from pathlib import Path
import py_compile, re, subprocess

if 'ROBOTWIN_VIDEO_PATCH' not in globals() or 'EVALUATOR_VIDEO_PATCH' not in globals():
    raise RuntimeError('Run the preceding patch definition cell first')
_base_task = PROJECT_ROOT / 'envs/_base_task.py'
_evaluator = EVALUATOR.resolve()

def _video_capture_state():
    base_source = _base_task.read_text()
    evaluator_source = _evaluator.read_text()
    base_ready = ('def _maybe_write_eval_video_frame' in base_source and
                  '_eval_video_sim_steps % 10 == 0' in base_source)
    evaluator_ready = (evaluator_source.count('task_env._write_eval_video_frame()') == 2 and
                       len(re.findall(r'"-framerate",\s*"25"', evaluator_source)) == 2)
    if ('def _maybe_write_eval_video_frame' in base_source and not base_ready):
        raise RuntimeError('An incompatible RoboTwin video patch is already present')
    if ('task_env._write_eval_video_frame()' in evaluator_source or
            re.search(r'"-framerate",\s*"25"', evaluator_source)) and not evaluator_ready:
        raise RuntimeError('The evaluator has a partial or incompatible video patch')
    return base_ready, evaluator_ready

base_ready, evaluator_ready = _video_capture_state()
pending = []
if not base_ready:
    pending.append((PROJECT_ROOT, ROBOTWIN_VIDEO_PATCH))
if not evaluator_ready:
    pending.append((_evaluator.parents[2], EVALUATOR_VIDEO_PATCH))
for source_root, patch_text in pending:
    checked = subprocess.run(['git', '-C', str(source_root), 'apply', '--check', '-'],
                             input=patch_text, text=True, capture_output=True)
    if checked.returncode:
        raise RuntimeError(f'Video patch does not match {source_root}: {checked.stderr.strip()}')
for source_root, patch_text in pending:
    subprocess.run(['git', '-C', str(source_root), 'apply', '-'],
                   input=patch_text, text=True, capture_output=True, check=True)
if _video_capture_state() != (True, True):
    raise RuntimeError('Continuous video capture was not fully enabled')
py_compile.compile(str(_base_task), doraise=True)
py_compile.compile(str(_evaluator), doraise=True)
print('Continuous video capture ready' + (' (updated)' if pending else ' (already installed)'))


In [ ]:
import subprocess
from video_files import find_video

VIDEO_PLAYBACK_SPEED = 5  # Use 1 to display the original recording speed.
if not isinstance(VIDEO_PLAYBACK_SPEED, int) or VIDEO_PLAYBACK_SPEED < 1:
    raise ValueError('VIDEO_PLAYBACK_SPEED must be a positive integer')

base_task_source = PROJECT_ROOT / 'envs/_base_task.py'
if ('_eval_video_sim_steps % 10 == 0' not in base_task_source.read_text() or
        EVALUATOR.read_text().count('task_env._write_eval_video_frame()') != 2):
    raise RuntimeError('Run the preceding video setup cell before recording')

method_order = ['robotwin']
method_names = {'robotwin': 'Official RoboTwin checkpoint'}
video_found = False
for method in method_order:
    candidates = []
    for scene in ('clean', 'random'):
        result_root = RUN_DIR / 'evaluations/no_video'
        rows = read_rows(result_root / f'{method}_{scene}_episodes.jsonl')
        successful = [row for row in rows if row.get('success')]
        successful.sort(key=lambda row: int(row.get('rollout_steps') or 10**9))
        candidates += [(scene, int(row['seed'])) for row in successful]
    if not candidates:
        print(f'{method_names[method]} had no successful evaluated seed to record.')
        continue
    for scene, seed in candidates:
        tag = f'{method}_success_video_continuous_{scene}_seed{seed}'
        output = RUN_DIR / 'evaluations' / tag
        command = [sys.executable, str(SCRIPT_ROOT / 'eval/run_paired_eval.py'),
                   '--run-dir', str(RUN_DIR), '--result-tag', tag,
                   '--hip-gpu', str(GPU_IDS[0]), '--port', str(PORT + 10),
                   '--count', '1', '--steps', str(STEPS), '--task-name', TASK_NAME,
                   '--method', method, '--scene', scene, '--seed', str(seed), '--video']
        def video_snapshot(elapsed):
            rows = read_rows(output / f'{method}_{scene}_episodes.jsonl')
            state = ('succeeded' if rows[0].get('success') else 'did not succeed') if rows else 'loading model and running episode'
            return f'{scene} seed {seed}: {state}; elapsed {duration(elapsed)}.'
        started = time.monotonic()
        run_monitored(command, RUN_DIR / f'{tag}.log',
                      f'{method_names[method]} success video: {scene} seed {seed}', video_snapshot)
        record_time(tag, started)
        rows = read_rows(output / f'{method}_{scene}_episodes.jsonl')
        if len(rows) != 1 or int(rows[0]['seed']) != seed:
            raise RuntimeError(f'Invalid video episode record: {output}')
        if rows[0]['success']:
            video_path = find_video(output / 'rollouts' / f'{method}_{scene}', rows[0])
            display_path = video_path
            if VIDEO_PLAYBACK_SPEED != 1:
                preview_dir = RUN_DIR / 'evaluations/video_previews'
                preview_dir.mkdir(parents=True, exist_ok=True)
                display_path = preview_dir / f'{tag}_{VIDEO_PLAYBACK_SPEED}x.mp4'
                if (not display_path.is_file() or
                        display_path.stat().st_mtime < video_path.stat().st_mtime):
                    subprocess.run([
                        'ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
                        '-i', str(video_path), '-vf',
                        f'setpts=PTS/{VIDEO_PLAYBACK_SPEED},fps=25',
                        '-an', '-c:v', 'libx264', '-preset', 'veryfast',
                        '-crf', '23', '-pix_fmt', 'yuv420p',
                        '-movflags', '+faststart', str(display_path),
                    ], check=True)
            display(Markdown(f'### {method_names[method]}: {scene} seed {seed} succeeded'))
            display(Video(filename=str(display_path), embed=True, mimetype='video/mp4'))
            print(f'displayed video ({VIDEO_PLAYBACK_SPEED}x):', display_path)
            video_found = True
            break
    if video_found:
        break
if not video_found:
    display(Markdown('No recorded rerun succeeded; no failure video is displayed.'))


## What to do with these results

- Inspect `RUN_DIR/manifest.json`, `stage_times.json`, `train.log`, `convert.log`, and `evaluations/no_video/`. JSONL files retain each episode's seed, outcome, and duration.
- This run trains the action expert for only 100 steps on 50 demonstrations from one task, without depth/video teachers. The official checkpoint's configuration is a longer, broader reference. Improving success requires revisiting the data and training budget and then using a formal evaluation.
- If `/workspace/runtime` points to the container's root disk, save any checkpoint, result, or video you need before recreating the container.
